# 01 — **Data Cleaning & Preparation**

**Dataset:** [UK E-Commerce Data](https://www.kaggle.com/datasets/carrie1/ecommerce-data) (Kaggle: `carrie1/ecommerce-data`)
**Author:** Ayush, Data Analyst
**Last updated:** 2026-09-05

## Overview

This notebook loads the raw transactional dataset, diagnoses data quality issues, and produces a clean, analysis-ready dataset for downstream notebooks:

- RFM segmentation
- Cohort analysis
- Revenue trend analysis

## 1. Load Raw Data

Importing the required Python libraries and load the dataset into a pandas DataFrame. The data will then be examined and prepared for further analysis and will be encoded accordingly 

In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
import seaborn as sns 
from charset_normalizer import from_path
import warnings 
warnings.filterwarnings('ignore')

In [2]:
result = from_path(r'D:\ProProjects\data-analytics-projects\ecom\dataset\data.csv').best()
print(result.encoding)

cp1250


In [3]:
df_raw = pd.read_csv(r'D:\ProProjects\data-analytics-projects\ecom\dataset\data.csv' , encoding = 'cp1250')
df_raw.sample()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
507919,579187,85176,SEWING SUSAN 21 NEEDLE SET,1,11/28/2011 15:31,1.63,NaN,United Kingdom


In [4]:
df_raw.columns = df_raw.columns.astype(str).str.lower()
df_raw.columns

Index(['invoiceno', 'stockcode', 'description', 'quantity', 'invoicedate',
       'unitprice', 'customerid', 'country'],
      dtype='object')

In [5]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   invoiceno    541909 non-null  object 
 1   stockcode    541909 non-null  object 
 2   description  540455 non-null  object 
 3   quantity     541909 non-null  int64  
 4   invoicedate  541909 non-null  object 
 5   unitprice    541909 non-null  float64
 6   customerid   406829 non-null  float64
 7   country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


## 2. Initial data quality audit

Quantifying every issue so cleaning decisions are evidence-based rather than assumption based

In [8]:
audit = {}

audit['total_rows'] = len(df_raw)
audit['missing_customerid'] = df_raw['customerid'].isna().sum()
audit['missing_description'] = df_raw['description'].isna().sum()
audit['negative_or_zero_quantity'] = (df_raw['quantity'] <= 0).sum()
audit['negative_or_zero_unitPprice'] = (df_raw['unitprice'] <= 0).sum()
audit['cancelled_invoices'] = df_raw['invoiceno'].astype(str).str.startswith('C').sum()
audit['duplicate_rows'] = df_raw.duplicated().sum()
audit['unique_customers'] = df_raw['customerid'].nunique()
audit['unique_products'] = df_raw['stockcode'].nunique()
audit['unique_countries'] = df_raw['country'].nunique()
audit['unique_invoices'] = df_raw['invoiceno'].nunique()

pd.Series(audit)

total_rows                     541909
missing_customerid             135080
missing_description              1454
negative_or_zero_quantity       10624
negative_or_zero_unitPprice      2517
cancelled_invoices               9288
duplicate_rows                   5268
unique_customers                 4372
unique_products                  4070
unique_countries                   38
unique_invoices                 25900
dtype: int64

In [9]:
# Checking how a cancellation actually look like?
df_raw[df_raw['invoiceno'].astype(str).str.startswith('C')].head(3)

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
141,C536379,D,Discount,-1,12/1/2010 9:41,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,12/1/2010 9:49,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,12/1/2010 10:24,1.65,17548.0,United Kingdom


In [11]:
# Checking how do rows with missing CustomerID look like? Are they valid transactions?
df_raw[df_raw['customerid'].isna()].head(3)

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
622,536414,22139,NaN,56,12/1/2010 11:52,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,12/1/2010 14:32,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,12/1/2010 14:32,2.51,NaN,United Kingdom


In [14]:
# Check for non-numeric stockcodes that may represent non-product entries.
odd_codes = df_raw[~df_raw['stockcode'].astype(str).str.contains(r'^\d')]['stockcode'].value_counts()
odd_codes.head(15)

stockcode
POST            1256
DOT              710
M                571
C2               144
D                 77
S                 63
BANK CHARGES      37
AMAZONFEE         34
CRUK              16
DCGSSGIRL         13
DCGSSBOY          11
gift_0001_20      10
gift_0001_10       9
gift_0001_30       8
DCGS0003           5
Name: count, dtype: int64

In [16]:
((df_raw['quantity'] < 0) & (~df_raw['invoiceno'].astype(str).str.startswith('C'))).sum()

np.int64(1336)

In [20]:
df_raw[((df_raw['quantity'] < 0) & (~df_raw['invoiceno'].astype(str).str.startswith('C')))].head(3)

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
2406,536589,21777,NaN,-10,12/1/2010 16:50,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,12/2/2010 14:42,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,12/3/2010 15:30,0.0,NaN,United Kingdom


In [21]:
((df_raw['unitprice'] < 0) & (~df_raw['invoiceno'].astype(str).str.startswith('C'))).sum()

np.int64(2)

In [22]:
df_raw[((df_raw['unitprice'] < 0) & (~df_raw['invoiceno'].astype(str).str.startswith('C')))].head(3)

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
299983,A563186,B,Adjust bad debt,1,8/12/2011 14:51,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,8/12/2011 14:52,-11062.06,NaN,United Kingdom


In [24]:
df_raw['country'].unique()

array(['United Kingdom', 'France', 'Australia', 'Netherlands', 'Germany',
       'Norway', 'EIRE', 'Switzerland', 'Spain', 'Poland', 'Portugal',
       'Italy', 'Belgium', 'Lithuania', 'Japan', 'Iceland',
       'Channel Islands', 'Denmark', 'Cyprus', 'Sweden', 'Austria',
       'Israel', 'Finland', 'Bahrain', 'Greece', 'Hong Kong', 'Singapore',
       'Lebanon', 'United Arab Emirates', 'Saudi Arabia',
       'Czech Republic', 'Canada', 'Unspecified', 'Brazil', 'USA',
       'European Community', 'Malta', 'RSA'], dtype=object)

## 3. Cleaning decisions

On the basis of current analysis , the cleaning decision are supposed to be made : 

| Issue | Decision | Rationale |
|---|---|---|
| `InvoiceDate` stored as text | Parse to `datetime64` | Required for any time-based analysis (cohorts, trends) |
| Cancellations (`InvoiceNo` starts with `C`) | Split into a **separate** `df_cancellations` frame, excluded from revenue/RFM analysis | These represent returns, not sales — mixing them in understates true demand and creates negative-revenue noise. Kept separately since return-rate is itself a useful metric |
| `Quantity <= 0` (non-cancellation) | Drop | Represents data entry errors or free samples with no valid quantity; not real sales |
| `UnitPrice <= 0` | Drop | Represents free items, adjustments, or errors (e.g. `StockCode` values like `POST`, `BANK CHARGES`, `AMAZONFEE`, `M`) rather than genuine product sales |
| Missing `CustomerID` | Drop for **customer-level** analysis (RFM, cohorts), but keep a copy for revenue-trend analysis where the customer isn't needed | ~25% of rows lack a CustomerID — likely guest/offline checkouts. Dropping them entirely would understate total revenue; but they can't be attributed to any customer, so segmentation/cohort work must exclude them |
| Exact duplicate rows | Drop | Same invoice/product/quantity/timestamp repeated — almost certainly a logging duplicate, not two genuine line items |
| `Description` missing | Fill with `'UNKNOWN'` for retained rows | Doesn't affect revenue math; only cosmetic for product-level breakdowns |
| `Country` missing | Already filled with `'Unspecified'` | Doesn't affect revenue math; only cosmetic for product-level breakdowns |

We create a `TotalPrice = Quantity * UnitPrice` column, since that's the actual line-item revenue and is used everywhere downstream.

In [25]:
df = df_raw.copy()

df['invoicedate'] = pd.to_datetime(df['invoicedate'], format='mixed')

df['iscancellation'] = df['invoiceno'].astype(str).str.startswith('C')
df_cancellations = df[df['iscancellation']].copy()
df = df[~df['iscancellation']].copy()

print(f"Cancellation rows set aside: {len(df_cancellations):,}")
print(f"Remaining rows: {len(df):,}")

Cancellation rows set aside: 9,288
Remaining rows: 532,621


In [26]:
# Drop invalid quantity/price rows (data entry errors / non-product line items)
before = len(df)
df = df[(df['quantity'] > 0) & (df['unitprice'] > 0)]
print(f"Dropped {before - len(df):,} rows with Quantity<=0 or UnitPrice<=0")
print(f"Remaining rows: {len(df):,}")

Dropped 2,517 rows with Quantity<=0 or UnitPrice<=0
Remaining rows: 530,104


In [27]:
# Drop exact duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df):,} exact duplicate rows")
print(f"Remaining rows: {len(df):,}")

Dropped 5,226 exact duplicate rows
Remaining rows: 524,878


In [28]:
# Fill missing descriptions (cosmetic only — doesn't affect revenue)
df['description'] = df['description'].fillna('unknown').str.strip()
# Derived revenue column
df['totalprice'] = df['quantity'] * df['unitprice']
df.head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,iscancellation,totalprice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34


In [30]:
# Create datasets for revenue and customer analysis
# All valid sales, including guest checkouts
df_full_revenue = df.copy()
# Sales with known CustomerID for customer analysis
df_customer = df[df['customerid'].notna()].copy()
df_customer['customerid'] = df_customer['customerid'].astype(int)

print(f"df_full_revenue: {df_full_revenue.shape}")
print(f"df_customer:     {df_customer.shape}  ({df_customer['customerid'].nunique():,} unique customers)")

df_full_revenue: (524878, 10)
df_customer:     (392692, 10)  (4,338 unique customers)


## 4. Sanity checks on the cleaned data

In [31]:
print("Date range:", df_full_revenue['invoicedate'].min(), "to", df_full_revenue['invoicedate'].max())
print("Total revenue (cleaned, all rows):   £{:,.2f}".format(df_full_revenue['totalprice'].sum()))
print("Total revenue (customer-only subset): £{:,.2f}".format(df_customer['totalprice'].sum()))
print("Countries:", df_full_revenue['country'].nunique())
df_full_revenue.describe(include='all').T

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Total revenue (cleaned, all rows):   £10,642,110.80
Total revenue (customer-only subset): £8,887,208.89
Countries: 38


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
invoiceno,524878,19960,573585,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
stockcode,524878,3922,85123A,2253,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,524878,4015,WHITE HANGING HEART T-LIGHT HOLDER,2311,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quantity,524878.0,NaN,NaN,NaN,10.6166,1.0,1.0,4.0,11.0,80995.0,156.280031
invoicedate,524878,NaN,NaN,NaN,2011-07-04 15:30:16.317049088,2010-12-01 08:26:00,2011-03-28 12:13:00,2011-07-20 11:22:00,2011-10-19 11:41:00,2011-12-09 12:50:00,NaN
unitprice,524878.0,NaN,NaN,NaN,3.922573,0.001,1.25,2.08,4.13,13541.33,36.093028
customerid,392692.0,NaN,NaN,NaN,15287.843865,12346.0,13955.0,15150.0,16791.0,18287.0,1713.539549
country,524878,38,United Kingdom,479985,NaN,NaN,NaN,NaN,NaN,NaN,NaN
iscancellation,524878,1,False,524878,NaN,NaN,NaN,NaN,NaN,NaN,NaN
totalprice,524878.0,NaN,NaN,NaN,20.275399,0.001,3.9,9.92,17.7,168469.6,271.693566


In [32]:
# Quick check: cancellation value, for context (not merged back in, but useful to report)
df_cancellations['totalprice'] = df_cancellations['quantity'] * df_cancellations['unitprice']
print(f"Cancellation line items: {len(df_cancellations):,}")
print(f"Cancellation value (abs): £{df_cancellations['totalprice'].abs().sum():,.2f}")

Cancellation line items: 9,288
Cancellation value (abs): £896,812.49


## 5. Persist cleaned datasets

Saved to `outputs/data/` so every downstream notebook loads a single, consistent, already-cleaned source instead of repeating this logic.

In [35]:
from pathlib import Path

# Create output directory
Path("outputs/data").mkdir(parents=True, exist_ok=True)

# Save cleaned datasets
df_full_revenue.to_csv("outputs/data/cleaned_full_revenue.csv", index=False)
df_customer.to_csv("outputs/data/cleaned_customer.csv", index=False)
df_cancellations.to_csv("outputs/data/cancellations.csv", index=False)

print("Saved:")
print(" - outputs/data/cleaned_full_revenue.csv")
print(" - outputs/data/cleaned_customer.csv")
print(" - outputs/data/cancellations.csv")

Saved:
 - outputs/data/cleaned_full_revenue.csv
 - outputs/data/cleaned_customer.csv
 - outputs/data/cancellations.csv


In [38]:
print("SUMMARY OF THIS NOTEBOOK")
print(f"- Started with {len(df_raw):,} raw rows.")
print("- Removed cancellations (returns), invalid quantity/price rows, and exact duplicates.")
print(f"- Result: {len(df_full_revenue):,} clean sales line items covering "
      f"{df_customer['customerid'].nunique():,} unique customers across {df_full_revenue['country'].nunique():,} countries.")
print("- Two datasets exported for downstream use: full revenue (incl. guest checkouts) and customer-identified subset (RFM/cohorts).")

SUMMARY OF THIS NOTEBOOK
- Started with 541,909 raw rows.
- Removed cancellations (returns), invalid quantity/price rows, and exact duplicates.
- Result: 524,878 clean sales line items covering 4,338 unique customers across 38 countries.
- Two datasets exported for downstream use: full revenue (incl. guest checkouts) and customer-identified subset (RFM/cohorts).
